In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
arquivo = r'C:\Users\Matheus\Documents\Trabalho Final - Ciência de Dados\train_sujo.csv'

In [3]:
df = pd.read_csv(arquivo)

In [4]:
display(df)

,Employee ID,Date of Joining,Gender,Company Type,WFH Setup Available,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate
0,fffe32003400330031003000,08-13-2008,m,SERVICE,Y,2.0,3.0,4.3,0.36
1,fffe32003600380032003800,03-11-2008,m,Service,Sim,2.0,4.0,5.0,0.36
2,fffe32003900330031003700,2008-02-13,male,Service,N,3.0,5.0,7.1,0.68
3,fffe31003100340030003800,10-Aug-08,Male,serv.,Nao,4.0,8.0,6.6,0.54
4,fffe32003900320031003800,26/05/2008,Female,Product,N,4.0,7.0,7.5,0.61
...,...,...,...,...,...,...,...,...,...
23883,fffe32003900380034003400,04/04/2008,F,PRODUCT,Sim,3.0,5.0,6.8,0.58
23884,fffe32003600360039003200,24/03/2008,Mulher,Service,No,2.0,4.0,6.6,0.57
23885,fffe3600380039003800,08-30-2008,Mulher,Service,NO,1.0,2.0,5.6,0.40
23886,fffe3700330039003100,10/07/2008,female,PRODUCT,N,0.0,1.0,4.0,NaN


In [5]:
df.dtypes

Employee ID              object
Date of Joining          object
Gender                   object
Company Type             object
WFH Setup Available      object
Designation             float64
Resource Allocation     float64
Mental Fatigue Score    float64
Burn Rate               float64
dtype: object

In [6]:
print("Contagem total de registros:", df.count())

print("\nNulos por coluna:")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "  Nenhum nulo encontrado.")

Contagem total de registros: Employee ID             23888
Date of Joining         22932
Gender                  22932
Company Type            22932
WFH Setup Available     22932
Designation             22932
Resource Allocation     21544
Mental Fatigue Score    20881
Burn Rate               21812
dtype: int64

Nulos por coluna:
Date of Joining          956
Gender                   956
Company Type             956
WFH Setup Available      956
Designation              956
Resource Allocation     2344
Mental Fatigue Score    3007
Burn Rate               2076
dtype: int64


In [7]:
df.describe()

,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate
count,22932.000000,21544.000000,20881.000000,21812.000000
mean,2.176391,4.482455,6.500158,0.452085
std,1.133985,2.043433,8.625630,0.197882
min,0.000000,1.000000,-5.000000,0.000000
25%,1.000000,3.000000,4.500000,0.320000
50%,2.000000,4.000000,5.900000,0.450000
75%,3.000000,6.000000,7.100000,0.590000
max,5.000000,10.000000,99.900000,1.000000


In [8]:
df.columns

Index(['Employee ID', 'Date of Joining', 'Gender', 'Company Type',
       'WFH Setup Available', 'Designation', 'Resource Allocation',
       'Mental Fatigue Score', 'Burn Rate'],
      dtype='object')

In [9]:
df = pd.read_csv(arquivo)

In [10]:
df.dropna(subset=['Burn Rate'], inplace=True)

df['Date of Joining'] = pd.to_datetime(df['Date of Joining'], errors='coerce')
df['Date of Joining'] = df['Date of Joining'].fillna(df['Date of Joining'].median())

df['Designation'] = pd.to_numeric(df['Designation'], errors='coerce')
df['Resource Allocation'] = pd.to_numeric(df['Resource Allocation'], errors='coerce')
df['Mental Fatigue Score'] = pd.to_numeric(df['Mental Fatigue Score'], errors='coerce')

df.loc[~df['Mental Fatigue Score'].between(0, 10), 'Mental Fatigue Score'] = np.nan

cols_texto = ['Gender', 'Company Type', 'WFH Setup Available']

df[cols_texto] = df[cols_texto].apply(
    lambda coluna: coluna.astype(str).str.strip().str.upper()
)

df['Gender'] = df['Gender'].replace({
    'F': 'FEMALE',
    'MULHER': 'FEMALE',
    'FEMININO': 'FEMALE',
    'M': 'MALE',
    'HOMEM': 'MALE',
    'MASCULINO': 'MALE'
})

df['Company Type'] = df['Company Type'].replace({
    'SERV.': 'SERVICE',
    'SERVICO': 'SERVICE',
    'SERVIÇO': 'SERVICE',
    'PROD.': 'PRODUCT',
    'PRODUTO': 'PRODUCT'
})

df['WFH Setup Available'] = df['WFH Setup Available'].replace({
    'Y': 'YES',
    '1': 'YES',
    '1.0': 'YES',
    'SIM': 'YES',
    'S': 'YES',
    'N': 'NO',
    '0': 'NO',
    '0.0': 'NO',
    'NAO': 'NO',
    'NÃO': 'NO'
})

df['Designation'] = df['Designation'].fillna(df['Designation'].mean())
df['Resource Allocation'] = df['Resource Allocation'].fillna(df['Resource Allocation'].mean())
df['Mental Fatigue Score'] = df['Mental Fatigue Score'].fillna(df['Mental Fatigue Score'].mean())

df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Company Type'] = df['Company Type'].fillna(df['Company Type'].mode()[0])
df['WFH Setup Available'] = df['WFH Setup Available'].fillna(df['WFH Setup Available'].mode()[0])

In [12]:
df['WFH Setup Available'] = (
    df['WFH Setup Available']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        'YES': 1,
        'Y': 1,
        'SIM': 1,
        'S': 1,
        '1': 1,
        '1.0': 1,

        'NO': 0,
        'N': 0,
        'NAO': 0,
        'NÃO': 0,
        '0': 0,
        '0.0': 0,

        'NAN': np.nan,
        'NONE': np.nan,
        '': np.nan
    })
)

df['WFH Setup Available'] = pd.to_numeric(df['WFH Setup Available'], errors='coerce')
df['WFH Setup Available'] = df['WFH Setup Available'].fillna(df['WFH Setup Available'].mode()[0])


df['Gender'] = (
    df['Gender']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        'FEMALE': 0,
        'F': 0,
        'MULHER': 0,
        'FEMININO': 0,

        'MALE': 1,
        'M': 1,
        'HOMEM': 1,
        'MASCULINO': 1,

        'NAN': np.nan,
        'NONE': np.nan,
        '': np.nan
    })
)

df['Gender'] = pd.to_numeric(df['Gender'], errors='coerce')
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])


df['Company Type'] = (
    df['Company Type']
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        'SERVICE': 0,
        'SERV.': 0,
        'SERVICO': 0,
        'SERVIÇO': 0,

        'PRODUCT': 1,
        'PROD.': 1,
        'PRODUTO': 1,

        'NAN': np.nan,
        'NONE': np.nan,
        '': np.nan
    })
)

df['Company Type'] = pd.to_numeric(df['Company Type'], errors='coerce')
df['Company Type'] = df['Company Type'].fillna(df['Company Type'].mode()[0])

C:\Users\Matheus\AppData\Local\Temp\ipykernel_35392\1700507220.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({
C:\Users\Matheus\AppData\Local\Temp\ipykernel_35392\1700507220.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({
C:\Users\Matheus\AppData\Local\Temp\ipykernel_35392\1700507220.py:62: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behav

In [13]:
display(df)

,Employee ID,Date of Joining,Gender,Company Type,WFH Setup Available,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate
0,fffe32003400330031003000,2008-08-13,1.0,0.0,1.0,2.0,3.000000,4.3,0.36
1,fffe32003600380032003800,2008-03-11,1.0,0.0,1.0,2.0,4.000000,5.0,0.36
2,fffe32003900330031003700,2008-07-07,1.0,0.0,0.0,3.0,5.000000,7.1,0.68
3,fffe31003100340030003800,2008-07-07,1.0,0.0,0.0,4.0,8.000000,6.6,0.54
4,fffe32003900320031003800,2008-07-07,0.0,1.0,0.0,4.0,7.000000,7.5,0.61
...,...,...,...,...,...,...,...,...,...
23881,fffe31003600300030003200,2008-07-07,0.0,0.0,0.0,2.0,4.000000,7.9,0.58
23883,fffe32003900380034003400,2008-07-07,0.0,1.0,1.0,3.0,5.000000,6.8,0.58
23884,fffe32003600360039003200,2008-07-07,0.0,0.0,0.0,2.0,4.000000,6.6,0.57
23885,fffe3600380039003800,2008-08-30,0.0,0.0,0.0,1.0,2.000000,5.6,0.40


In [15]:
var_indep = ['Gender', 'Company Type',
       'WFH Setup Available', 'Designation', 'Resource Allocation',
       'Mental Fatigue Score']

var_dep = ['Burn Rate']

df_x = df[var_indep]
df_y = df[var_dep]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(df_x, df_y, test_size=0.2, random_state=42) 

print(X_train.shape[0])
print(y_train.shape[0])

17449
17449


In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

linear_model = LinearRegression()
decision_tree = DecisionTreeRegressor(random_state=42)
random_forest = RandomForestRegressor(random_state=42)

In [18]:
linear_model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1, 6)","[[ 0.01,-0. ,-0.02, 0.02, 0.03, 0.06]]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](6,)","['Gender','Company Type','WFH Setup Available','Designation', 'Resource Allocation','Mental Fatigue Score']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float64](1,)",[-0.09]
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,6
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,6


In [19]:
decision_tree.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes 

In [20]:
random_forest.fit(X_train, y_train)

c:\Users\Matheus\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [21]:
predict_linear = linear_model.predict(X_test)

In [22]:
predict_tree = decision_tree.predict(X_test)

In [23]:
predict_forest = random_forest.predict(X_test)

In [24]:
mae_linear = mean_absolute_error(y_test, predict_linear)
mae_tree = mean_absolute_error(y_test, predict_tree)
mae_forest = mean_absolute_error(y_test, predict_forest)

r_squared_linear = r2_score(y_test, predict_linear)
r_squared_tree = r2_score(y_test, predict_tree)
r_squared_forest = r2_score(y_test, predict_forest)

print("linear: ", mae_linear, r_squared_linear)
print("tree: ", mae_tree, r_squared_tree)
print("forest: ", mae_forest, r_squared_forest)

linear:  0.05952205537554793 0.8361039589617615
tree:  0.05518212689209334 0.862426216162009
forest:  0.05267014029852232 0.8756269557897776


In [25]:
arquivo_teste = r'C:\Users\Matheus\Documents\Trabalho Final - Ciência de Dados\test_sujo.csv'
df_teste = pd.read_csv(arquivo_teste)
display(df_teste)

,Employee ID,Date of Joining,Gender,Company Type,WFH Setup Available,Designation,Resource Allocation,Mental Fatigue Score
0,fffe32003500320038003600,12/02/2008,m,produto,y,3.0,7.0,5.7
1,fffe3200320038003200,02-Apr-08,Homem,servico,N,1.0,2.0,NaN
2,fffe32003400390034003200,2008-06-22,MALE,serv.,YES,0.0,1.0,1.6
3,fffe32003900310036003300,27-Jul-08,Mulher,Service,NO,2.0,5.0,7.3
4,fffe31003300310033003700,2008-02-27,Female,Service,0,1.0,3.0,6.9
...,...,...,...,...,...,...,...,...
12857,fffe37003000,19/02/2008,Homem,serv.,n,4.0,8.0,8.1
12858,fffe32003200330031003400,29/07/2008,F,Service,YES,2.0,4.0,5.6
12859,fffe33003100390036003400,2008-01-11,M,PRODUCT,Y,4.0,7.0,6.0
12860,fffe33003200360033003200,04-05-2008,f,Product,No,3.0,5.0,7.9


In [26]:
arquivo_teste = r'C:\Users\Matheus\Documents\Trabalho Final - Ciência de Dados\test.csv'
df_teste = pd.read_csv(arquivo_teste)

df_teste['Designation'] = df_teste['Designation'].fillna(df_teste['Designation'].mean())
df_teste['Resource Allocation'] = df_teste['Resource Allocation'].fillna(df_teste['Resource Allocation'].mean())
df_teste['Mental Fatigue Score'] = df_teste['Mental Fatigue Score'].fillna(df_teste['Mental Fatigue Score'].mean())

df_teste['WFH Setup Available'] = df_teste['WFH Setup Available'].str.upper()
df_teste['WFH Setup Available'] = df_teste['WFH Setup Available'].str.replace('NO', '0').str.replace('YES', '1')
df_teste['WFH Setup Available'] = pd.to_numeric(df_teste['WFH Setup Available'])

df_teste['Gender'] = df_teste['Gender'].str.upper()
df_teste['Gender'] = df_teste['Gender'].str.replace('FEMALE', '0').str.replace('MALE', '1')
df_teste['Gender'] = pd.to_numeric(df_teste['Gender'])

df_teste['Company Type'] = df_teste['Company Type'].str.upper()
df_teste['Company Type'] = df_teste['Company Type'].str.replace('SERVICE', '0').str.replace('PRODUCT', '1')
df_teste['Company Type'] = pd.to_numeric(df_teste['Company Type'])

df_teste.to_csv(r'C:\Users\Matheus\Documents\Trabalho Final - Ciência de Dados\test_limpo.csv', index=False)

ids_funcionarios = df_teste['Employee ID']
X_teste = df_teste[var_indep]

predict_teste = random_forest.predict(X_teste)

def classificar_risco(nota):
    if nota < 0.3:
        return 'Baixo'
    elif nota < 0.6:
        return 'Medio'
    else:
        return 'Alto'

df_result = pd.DataFrame({
    'Employee ID': ids_funcionarios,
    'Burn Rate previsto': predict_teste
})

df_result['Risco'] = df_result['Burn Rate previsto'].apply(classificar_risco)
df_result.to_csv(r'C:\Users\Matheus\Documents\Trabalho Final - Ciência de Dados\predicao.csv', index=False)
display(df_result)

,Employee ID,Burn Rate previsto,Risco
0,fffe31003300390039003000,0.605293,Alto
1,fffe31003300310037003800,0.345664,Medio
2,fffe33003400380035003900,0.451672,Medio
3,fffe3100370039003200,0.363671,Medio
4,fffe32003600390036003700,0.583977,Medio
...,...,...,...
12245,fffe3900310034003700,0.443311,Medio
12246,fffe32003600330034003000,0.407009,Medio
12247,fffe31003800340039003000,0.833700,Alto
12248,fffe32003600380031003800,0.586917,Medio
